In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
import torch
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.preprocessing import StandardScaler
import joblib
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import optuna

# Functions

In [2]:
def naive_forecaster(X_test):
    lag1_cols = [f'DA_price lag1_hour{i}' for i in range(24)]
    lag7_cols = [f'DA_price lag7_hour{i}' for i in range(24)]
    prediction = []
    for index, row in X_test.iterrows():
        if pd.to_datetime(index).weekday() in [1,2,3,4,6]:
            prediction.append(row[lag1_cols].values)
        else:
            prediction.append(row[lag7_cols].values)
    prediction = np.array(prediction)
    return prediction

# calculate smape
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

#day average error
def dae(y_true, y_pred):
    return np.mean(np.abs(np.mean(y_pred,axis=1) - np.mean(y_true,axis=1)))

def rmae(y_true, y_pred, y_pred_naive):
    mae_pred = np.mean(np.abs(y_pred - y_true))
    mae_naive = np.mean(np.abs(y_pred_naive - y_true))
    return mae_pred/mae_naive

In [3]:
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Assuming your data is in pandas DataFrames
def prepare_data(X_train, y_train, X_val, y_val, X_test):
    X_train, y_train = torch.FloatTensor(X_train.values).to(device), torch.FloatTensor(y_train.values).to(device)
    X_val, y_val = torch.FloatTensor(X_val.values).to(device), torch.FloatTensor(y_val.values).to(device)
    X_test2 = torch.FloatTensor(X_test.values).to(device)
    
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)
    
    return train_loader, val_loader, X_test2

In [4]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, output_size, hidden_layers, neurons_per_layer, reg_type, reg_strength):
        super(FeedForwardNN, self).__init__()
        layers = []
        last_layer_size = input_size
        for i in range(hidden_layers):
            layers.append(nn.Linear(last_layer_size, neurons_per_layer))
            layers.append(nn.ReLU())
            last_layer_size = neurons_per_layer
        layers.append(nn.Linear(last_layer_size, output_size))
        
        self.model = nn.Sequential(*layers)

        if reg_type == 'l1':
            self.regularization = nn.L1Loss()
        else:  # l2
            self.regularization = nn.MSELoss()
        self.reg_strength = reg_strength
        self.to(device)

    def forward(self, x):
        return self.model(x)
    
    def loss(self, output, target):
        # Adjust logcosh for multi-output
        logcosh_loss = torch.log(torch.cosh(output - target)).mean()
        reg_loss = sum(param.abs().sum() for param in self.parameters()) if self.regularization.__class__.__name__ == 'L1Loss' else sum(param.pow(2).sum() for param in self.parameters())
        return logcosh_loss + self.reg_strength * reg_loss

In [5]:
def train_model(model, train_loader, val_loader, epochs=50, patience=10):
    optimizer = optim.Adam(model.parameters())
    best_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            X, y = batch
            optimizer.zero_grad()
            output = model(X)
            loss = model.loss(output, y)
            loss.backward()
            optimizer.step()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                output = model(X_val)
                val_loss += model.loss(output, y_val).item()
        
        val_loss /= len(val_loader)
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return best_loss

In [6]:
def objective(trial, train_loader, val_loader, input_size):
    # Hyperparameters to optimize
    n_layers = trial.suggest_int('n_layers', 1, 4)
    neurons = trial.suggest_int('neurons', 32, 256)
    reg_type = trial.suggest_categorical('reg_type', ['l1', 'l2'])
    reg_strength = trial.suggest_float('reg_strength', 1e-5, 1e-1, log=True)
    
    model = FeedForwardNN(input_size=input_size, output_size=24, hidden_layers=n_layers, neurons_per_layer=neurons, reg_type=reg_type, reg_strength=reg_strength)
    loss = train_model(model, train_loader, val_loader)
    return loss

In [7]:
def get_best_params_and_test(country, period=0, n_trials=50):

    # Load the data
    inputs = pd.read_csv(os.path.join("cut_data", country, "inputs.csv"),index_col=0)
    inputs.fillna(value=0, inplace=True)
    outputs = pd.read_csv(os.path.join("cut_data", country, "outputs.csv"),index_col=0)
    outputs.fillna(value=0, inplace=True)
    
    # fill nan with ffil
    inputs = inputs.fillna(value=0)
    #outputs = outputs.ffill()

    X_train = inputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']
    y_train = outputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']

    X_val = inputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']
    y_val = outputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']

    X_test = inputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']
    y_test = outputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']
    
    # Generate the list of column names
    column_names = [f'DA_price lag1_hour{i}' for i in range(24)]

    # Get the indices of the columns
    column_indices = [X_train.columns.get_loc(name) for name in column_names if name in X_train.columns]


    # transform y_train_val, y_test by substraction of the X_train_val X_test column_indices values
    y_train2 = pd.DataFrame(y_train.values - X_train.iloc[:,column_indices].values, columns=y_train.columns, index=y_train.index)
    y_val2 = pd.DataFrame(y_val.values - X_val.iloc[:,column_indices].values, columns=y_val.columns, index=y_val.index)

    # Scaling
    scaler1, scaler2 = StandardScaler(), StandardScaler()

    X_train = pd.DataFrame(scaler1.fit_transform(X_train), columns=X_train.columns)
    X_val = pd.DataFrame(scaler1.transform(X_val), columns=X_val.columns)
    X_test2 = pd.DataFrame(scaler1.transform(X_test), columns=X_test.columns)

    y_train2 = pd.DataFrame(scaler2.fit_transform(y_train2), columns=y_train2.columns)
    y_val2 = pd.DataFrame(scaler2.transform(y_val2), columns=y_val2.columns)
    #y_test2 = pd.DataFrame(scaler2.transform(y_test2), columns=y_test2.columns)
    
    train_loader, val_loader, X_test2 = prepare_data(X_train, y_train2, X_val, y_val2, X_test2)

    # Hyperparameter optimization
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, train_loader, val_loader, X_train.shape[1]), n_trials=n_trials, n_jobs=-1)

    # Get best hyperparameters
    best_params = study.best_params
    
    # Train with best hyperparameters
    best_model = FeedForwardNN(input_size=X_train.shape[1], 
                               output_size=24, 
                               hidden_layers=best_params['n_layers'], 
                               neurons_per_layer=best_params['neurons'], 
                               reg_type=best_params['reg_type'], 
                               reg_strength=best_params['reg_strength'])

    best_loss = train_model(best_model, train_loader, val_loader)

    # Predict on test set
    best_model.eval()
    with torch.no_grad():
        predictions = best_model(X_test2).cpu().numpy()  # Move predictions back to CPU for numpy conversion
        
    y_pred2 = predictions
    y_pred_naive = naive_forecaster(X_test)
    y_test_val = y_test.values

    # inverse transform y_pred, y_pred_naive, y_test_val
    y_pred2 = scaler2.inverse_transform(y_pred2)
    y_pred_naive = scaler2.inverse_transform(y_pred_naive)
    y_test_val = scaler2.inverse_transform(y_test_val)
    
    y_pred = y_pred2 + X_test.iloc[:,column_indices].values

    smape_score = smape(y_test_val.flatten(), y_pred.flatten())
    mae_score = mean_absolute_error(y_test_val.flatten(), y_pred.flatten())
    dae_score = dae(y_test_val, y_pred)
    rmae_score = rmae(y_test_val, y_pred, y_pred_naive)

    naiv_smape = smape(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_mae = mean_absolute_error(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_dae =  dae(y_test_val, y_pred_naive)
    
    return [country, period, smape_score, mae_score, dae_score, rmae_score, naiv_smape, naiv_mae, naiv_dae, str(study.best_params)]

# Training and evaluation

In [ ]:
country_name_list = sorted(os.listdir(os.path.join('cut_data')))

final_results = {}

for country in tqdm(country_name_list):
    for period in [0,4]:
        results = get_best_params_and_test(country, period, n_trials=50)
        final_results[f"{country}_{period}"] = results
        # save as json
        with open('dnn_final_results.json', 'w') as f:
            json.dump(final_results, f)
    #print(f'{country} is done')

  9%|▉         | 2/22 [05:20<53:26, 160.31s/it]


KeyboardInterrupt: 

## Save results as csv file

In [3]:

with open('dnn_final_results.json', 'r') as f:
    final_results = json.load(f)
    
final_results_df = pd.DataFrame(final_results).T
final_results_df.columns = ['country', 'period', 'smape', 'mae', 'dae', 'rmae', 'naive_smape', 'naive_mae', 'naive_dae', 'best_params']
final_results_df.to_csv('dnn_final_results.csv', index=False)